# Build AIMS4PT Cpx Pressure Workflow

This notebook builds the pressure-model workflow, including model-pool setup, OOD diagnostics, pressure-deviation prediction


# Setup and input data

In [ ]:
from pathlib import Path
import sys

def _find_project_root(start=None):
    """Find the project root by walking upward from the current directory."""
    path = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (path, *path.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml above the current directory.")

PROJECT_ROOT = _find_project_root()
PAPER_DIR = PROJECT_ROOT / "paper"
CACHE_DIR = PAPER_DIR / ".cache"
IMAGES_DIR = PAPER_DIR / ".images"
DATA_DIR = PAPER_DIR / "data"
for _dir in (CACHE_DIR, IMAGES_DIR, DATA_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
import os
import time
import importlib
import matplotlib.pyplot as plt
from aims4pt.model_tools.model_registry import  get_models_initial_pools, print_all_registered_thermobarometry, ALL_MODELS_MODULES
from aims4pt.toolkit_utils import get_file_path

interested_MODULES = [
    # "aims4pt.model_tools.Putirka_08",
    # "aims4pt.model_tools.Neave_Putirka_17",
    # "aims4pt.model_tools.Petrelli20",
    # "aims4pt.model_tools.Higgins21",
    # "aims4pt.model_tools.Jorgenson22",
    # "aims4pt.model_tools.Agreda2024",
    # "aims4pt.model_tools.Wang_21",
    "aims4pt.model_tools.Chicchi23", # shap! slow
]
# This import affects the final OOD models.

for module in ALL_MODELS_MODULES:
    importlib.import_module(module)




calculate_shap = False # make sure this to False when not needed, as it takes time and memory

timestamp = time.strftime("%Y%m%d_%H%M")

In [ ]:
unseen_experiments_path = r"C:\Users\13493\Documents\Thermobarometers\Database\data\cross_check_data\cpx_liq_unseen_unique_testv2__2025-1-8.xlsx"
unseen_experiments_path = r"C:\Users\13493\Documents\Thermobarometers\Database\data\cross_check_data\unique_cpx_liq_auto_filtered.xlsx"# This version is not filtered by the 10 kbar threshold.
#change to your path

unseen_experiments_df = pd.read_excel(unseen_experiments_path)

#col names for unseen_experiments_df
cpx_names = ['SiO2_cpx', 'TiO2_cpx', 'Al2O3_cpx',
        'Cr2O3_cpx', 'FeO_cpx', 'MnO_cpx', 'MgO_cpx', 'NiO_cpx',
       'CaO_cpx', 'Na2O_cpx', 'K2O_cpx', 'P2O5_cpx']
liq_names = ['SiO2_liq', 'TiO2_liq', 'Al2O3_liq',
        'FeO_liq', 'MnO_liq', 'MgO_liq', 
       'CaO_liq', 'Na2O_liq', 'K2O_liq', 'P2O5_liq']
P_col = 'P (kbar)'
T_col = 'T (C)'


In [ ]:
from aims4pt.model_tools.CpxTBSelect import workflow_thermobarometry


## clean


In [7]:
print("Before cleaning, unseen_experiments_df shape:", unseen_experiments_df.shape)
cpx_total = unseen_experiments_df[cpx_names].sum(axis=1)
unseen_experiments_df = unseen_experiments_df[(cpx_total>98)&(cpx_total<102)].reset_index(drop=True)
print("After total, unseen_experiments_df shape:", unseen_experiments_df.shape)
from aims4pt.data_tools.compositions import cpx_calculation
cpx_params = cpx_calculation(unseen_experiments_df[cpx_names])
MdivT= cpx_params["(Ca+Fe+Mg)/Si"]
unseen_experiments_df = unseen_experiments_df[(MdivT>0.9)&(MdivT<1.1)].reset_index(drop=True)
print("After cpx MdivT, unseen_experiments_df shape:", unseen_experiments_df.shape)
# kd filter
from aims4pt.data_tools.equilibrium import kdEquilibrium_test
mask, kd_values = kdEquilibrium_test(
    unseen_experiments_df[cpx_names],
    unseen_experiments_df[liq_names],
    kd = 0.28,
    error = 0.08,
    mode='Fe-Mg',)
unseen_experiments_df = unseen_experiments_df[mask].reset_index(drop=True)
print("After kd filter, unseen_experiments_df shape:", unseen_experiments_df.shape)
liq_total = unseen_experiments_df[liq_names].sum(axis=1)
liq_total.describe()

Before cleaning, unseen_experiments_df shape: (517, 36)
After total, unseen_experiments_df shape: (497, 36)
After cpx MdivT, unseen_experiments_df shape: (441, 36)
After kd filter, unseen_experiments_df shape: (293, 36)


count    293.000000
mean      98.324335
std        2.691512
min       86.971390
25%       97.802000
50%       99.813648
75%      100.000000
max      100.660000
dtype: float64

In [8]:

cpx_unseen = unseen_experiments_df[cpx_names].copy()
liq_unseen = unseen_experiments_df[liq_names].copy()
meta_unseen = unseen_experiments_df[[P_col, T_col]].copy()
# stratified split
from sklearn.model_selection import StratifiedShuffleSplit

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
P_values = unseen_experiments_df[P_col]
target_bin_size = 50
num_bins = max(2, min(10, len(P_values) // target_bin_size))
P_binned = pd.qcut(P_values, q=num_bins, labels=False, duplicates="drop")
(strat_train_idx, strat_test_idx), = sss.split(unseen_experiments_df, P_binned)
training_unseen_id = unseen_experiments_df.iloc[strat_train_idx].index
testing_unseen_id = unseen_experiments_df.iloc[strat_test_idx].index

#random split
# from sklearn.model_selection import train_test_split
# training_unseen_id, testing_unseen_id = train_test_split(
#     unseen_experiments_df.index,
#     test_size=0.2,
#     random_state=42,
#     )


# Initial all thermobarometry models

In [ ]:
P_model_pool = get_models_initial_pools("P", "both" , False) # get all P models including cpx_only and cpx_liq
P_model_names = list(model.model_name for model in P_model_pool)
T_model_pool = get_models_initial_pools("T", "both" , False) # get all T models including cpx_only and cpx_liq
T_model_names = list(model.model_name for model in T_model_pool)


In [21]:
P_model_dict = {model.model_name: model for model in P_model_pool}
T_model_dict = {model.model_name: model for model in T_model_pool}

## Pressure SHAP feature importance

### Pressure model

In [35]:
first_time_run_shap = False
P_SHAP_df_dict = {}
P_feature_importance_dict = {}
import pickle
import aims4pt.model_tools.trained_model.SHAP as SHAP_package

# stratified sampling based on P distribution
from sklearn.model_selection import StratifiedShuffleSplit

if first_time_run_shap:
    for model, model_name in zip(P_model_pool, P_model_names):
        print(f"Processing model: {model_name}")
        if model.X_cpx_training is None:
            continue
        path = get_file_path(SHAP_package, f"{model_name}_P_shap_df_default.pkl")
        path_fi = get_file_path(SHAP_package, f"{model_name}_P_feature_importance_default.pkl")
        if os.path.exists(path) and os.path.exists(path_fi):
            # with open(path, "rb") as f:
            #     shap_df = pickle.load(f)
            # with open(path_fi, "rb") as f:
            #     feature_importance_df = pickle.load(f)
            # P_SHAP_df_dict[model_name] = shap_df
            # P_feature_importance_dict[model_name] = feature_importance_df
            print(f"SHAP data for model {model_name} already exists. Skipping calculation.")
            continue

        # sampling 20% of training data for SHAP calculation
        len_training = model.X_cpx_training.shape[0]
        strat_split = StratifiedShuffleSplit(
            n_splits=1, test_size=0.2, random_state=42
        )

        # create bins for P values
        P_values = model.X_cpx_training["P_kbar"]
        num_bins = max(2, min(10, int(len_training / 50)))
        P_binned = pd.cut(P_values, bins=num_bins, labels=False)
        (strat_train_idx, strat_test_idx), = strat_split.split(model.X_cpx_training, P_binned)
        sampled_indices = strat_test_idx
        
        X_cpx_sampled = model.X_cpx_training.iloc[sampled_indices]
        X_liq_sampled = model.X_liq_training.iloc[sampled_indices]

        shap_df, feature_importance_df, shap_values =model.shap_calculation(
            X_cpx_sampled, X_liq_sampled, background_data=model.X_cpx_training, bg_liq=model.X_liq_training,
                                    package_predict_func=True, sampling_bg=100)
        
        P_SHAP_df_dict[model_name] = shap_df
        P_feature_importance_dict[model_name] = feature_importance_df
        with open(path, "wb") as f:
            pickle.dump(shap_df, f)
        with open(path_fi, "wb") as f:
            pickle.dump(feature_importance_df, f)
else:
    for model_name in P_model_names:
        try:
            path = get_file_path(SHAP_package, f"{model_name}_P_shap_df_default.pkl")
            path_fi = get_file_path(SHAP_package, f"{model_name}_P_feature_importance_default.pkl")
            with open(path, "rb") as f:
                shap_df = pickle.load(f)
            with open(path_fi, "rb") as f:
                feature_importance_df = pickle.load(f)
        except FileNotFoundError:
            continue
        P_SHAP_df_dict[model_name] = shap_df
        P_feature_importance_dict[model_name] = feature_importance_df

        

C:\Users\13493\Documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Putirka, 2008 eq32d_T; eq32a_P (cpx_only)_P_shap_df_default.pkl
C:\Users\13493\Documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Putirka, 2008 eq32d_T; eq32a_P (cpx_only)_P_feature_importance_default.pkl
C:\Users\13493\Documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Putirka, 2008 eq32d_T; eq32b_P (cpx_only)_P_shap_df_default.pkl
C:\Users\13493\Documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Putirka, 2008 eq32d_T; eq32b_P (cpx_only)_P_feature_importance_default.pkl
C:\Users\13493\Documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Petrelli et al., 2020 (cpx_only)_P_shap_df_default.pkl
C:\Users\13493\Documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\SHAP\Petrelli et al., 2020 (cpx_only)_P_feature_importance_default.pkl
C:\Users\13493\Documents\my_analysis_tools\my_ana

## Pressure weighted OOD detector

### Pressure model

In [37]:

from aims4pt.statistic_tools.out_of_distribution import ThermobarometerOODFactory
import pickle
first_time_run_OOD_key_detector = True
import aims4pt.model_tools.trained_model.OOD_detectors as OOD_detectors_package
OOD_detectors_P_weighted_dict = {}
if first_time_run_OOD_key_detector:
    for model, model_name in zip(P_model_pool, P_model_names):
        print(f"Processing model: {model_name}")
        if model.X_cpx_training is None:
            continue
        path = get_file_path(OOD_detectors_package, f"{model_name}_P_weighted.pkl")
        if os.path.exists(path):
            with open(path, "rb") as f:
                detector = pickle.load(f)
            OOD_detectors_P_weighted_dict[model_name] = detector
            continue
        model.feature_importance_df = P_feature_importance_dict[model_name]
        model.set_key_features(mode="all")
        detector = ThermobarometerOODFactory(model, "key", weighted=True)
        model.OOD_detector = detector
        OOD_detectors_P_weighted_dict[model_name] = detector
        with open(path, "wb") as f:
            pickle.dump(detector, f)
else:
    for model_name in P_model_names:
        try:
            path = get_file_path(OOD_detectors_package, f"{model_name}_P_weighted.pkl")
            with open(path, "rb") as f:
                detector = pickle.load(f)
        except FileNotFoundError:
            continue
        OOD_detectors_P_weighted_dict[model_name] = detector
        

   

Processing model: Putirka, 2008 eq32d_T; eq32a_P (cpx_only)
Processing model: Putirka, 2008 eq32d_T; eq32b_P (cpx_only)
Processing model: Petrelli et al., 2020 (cpx_only)
C:\Users\13493\Documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\OOD_detectors\Petrelli et al., 2020 (cpx_only)_P_weighted.pkl
Processing model: Higgins et al., 2021 (cpx_only)
C:\Users\13493\Documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\OOD_detectors\Higgins et al., 2021 (cpx_only)_P_weighted.pkl
Processing model: Jorgenson et al., 2022 (cpx_only)
C:\Users\13493\Documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\OOD_detectors\Jorgenson et al., 2022 (cpx_only)_P_weighted.pkl
Processing model: Ágreda-López et al., 2024 (cpx_only)
C:\Users\13493\Documents\my_analysis_tools\my_analysis_tools\model_tools\trained_model\OOD_detectors\Ágreda-López et al., 2024 (cpx_only)_P_weighted.pkl
Processing model: Wang et al., 2021 (cpx_only)
C:\Users\13493\Documents\

c:\Users\13493\anaconda3\envs\thermobarometry\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\13493\anaconda3\envs\thermobarometry\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.5.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\13493\anaconda3\envs\thermobarometry\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.5.1

## Pressure deviation predictor

In [ ]:
first_time_run_deviation_fit = True


from aims4pt.model_tools.deviation_models import deviation_model_for_a_thermobarometer
import aims4pt.model_tools.trained_model.Deviation_functions as Deviation_functions_package
r2_collect_P = {}
r2_collect_T = {}
deviation_functions_P_dict = {}
deviation_functions_T_dict = {}
# indices = [4, -3, -2] # ((P_model_pool[i], P_model_names[i]) for i in indices)
calib_cpx = cpx_unseen.loc[training_unseen_id]
calib_P = meta_unseen.loc[training_unseen_id, P_col]
calib_T = meta_unseen.loc[training_unseen_id, T_col]
calib_liq = liq_unseen.loc[training_unseen_id]
if first_time_run_deviation_fit:
    for model, model_name in zip(P_model_pool, P_model_names):
        print(f"Processing deviation function for model: {model_name}")
        save_path = get_file_path(Deviation_functions_package, f"P_{model_name}_deviation_function_v_ind.pkl")
        if os.path.exists(save_path):
            with open(save_path, "rb") as f:
                deviation_function = pickle.load(f)
            deviation_functions_P_dict[model_name] = deviation_function
            r2_collect_P[model_name] = deviation_function.r2
            continue
        deviation_function = deviation_model_for_a_thermobarometer(model)
        ood_mask = np.zeros(len(calib_cpx), dtype=bool)
        # try:
        #     ood_detector = OOD_detectors_P_weighted_dict[model_name]
        #     ood_mask = ood_detector.is_ood(calib_cpx,calib_liq)
        # except KeyError:
        #     print(f"No OOD detector found for model: {model_name}, proceeding without OOD filtering.")
        
        deviation_function.fit_deviation_model(calib_cpx[~ood_mask],
                                                calib_P[~ood_mask],
                                                calib_liq[~ood_mask], n_iter=100)
                                
        deviation_functions_P_dict[model_name] = deviation_function
        model.deviation_function = deviation_function
        r2_collect_P[model_name] = deviation_function.r2
        with open(save_path, "wb") as f:
            pickle.dump(deviation_function, f)
    
    
    for model, model_name in zip(T_model_pool, T_model_names):
        print(f"Processing deviation function for model: {model_name}")
        save_path = get_file_path(Deviation_functions_package, f"T_{model_name}_deviation_function_v_ind.pkl")
        if os.path.exists(save_path):
            with open(save_path, "rb") as f:
                deviation_function = pickle.load(f)
            deviation_functions_T_dict[model_name] = deviation_function
            r2_collect_T[model_name] = deviation_function.r2
            continue
        deviation_function = deviation_model_for_a_thermobarometer(model)
        ood_mask = np.zeros(len(calib_cpx), dtype=bool)
        # try:
        #     ood_detector = OOD_detectors_T_weighted_dict[model_name]
        #     ood_mask = ood_detector.is_ood(calib_cpx,calib_liq)
        # except KeyError:
        #     print(f"No OOD detector found for model: {model_name}, proceeding without OOD filtering.")
        deviation_function.fit_deviation_model(calib_cpx[~ood_mask],
                                                calib_T[~ood_mask],
                                                calib_liq[~ood_mask], n_iter=100)
        deviation_functions_T_dict[model_name] = deviation_function
        model.deviation_function = deviation_function
        r2_collect_T[model_name] = deviation_function.r2
        with open(save_path, "wb") as f:
            pickle.dump(deviation_function, f)

else:

    for model_name in P_model_names:
        load_path = get_file_path(Deviation_functions_package, f"P_{model_name}_deviation_function_v_ind.pkl")
        with open(load_path, "rb") as f:
            deviation_function = pickle.load(f)
        deviation_functions_P_dict[model_name] = deviation_function
        r2_collect_P[model_name] = deviation_function.r2
    

    for model_name in T_model_names:
        load_path = get_file_path(Deviation_functions_package, f"T_{model_name}_deviation_function_v_ind.pkl")
        with open(load_path, "rb") as f:
            deviation_function = pickle.load(f)
        deviation_functions_T_dict[model_name] = deviation_function
        r2_collect_T[model_name] = deviation_function.r2

# print(pd.DataFrame.from_dict(r2_collect_P, orient='index', columns=['R²']))
# print(pd.DataFrame.from_dict(r2_collect_T, orient='index', columns=['R²']))